In [608]:
import altair as alt
import polars as pl

alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

In [609]:
zkraceni = {
    "K - adresa knihovny: ulice": "ulice",
    "K - adresa knihovny: PSČ": "psc",
    "K - adresa knihovny: město": "mesto",
    "K - adresa knihovny: kraj": "kraj",
}

In [610]:
df = (
    pl.read_excel("data/knihovny/Evidence knihoven-11122025.xlsx")
    .filter(pl.col("aktivní / zrušená (vyřazená z evidence)") == "A")
    .rename(zkraceni)
    .with_columns(
        (
            pl.col("ulice")
            + ", "
            + pl.col("mesto")
            + ", "
            + pl.col("psc")
            + ", "
            + pl.col("kraj")
            + ", Czechia"
        ).alias("search_query")
    )
    .select(
        pl.col(
            [
                "I - NÁZEV KNIHOVNY",
                "J - druh knihovny",
                "ulice",
                "psc",
                "mesto",
                "kraj",
                "search_query",
            ]
        )
    )
    .with_columns(pl.all().str.strip_chars())
    .with_columns(pl.col("J - druh knihovny").str.to_lowercase())
)

Could not determine dtype for column 7, falling back to string
Could not determine dtype for column 9, falling back to string
Could not determine dtype for column 10, falling back to string
Could not determine dtype for column 20, falling back to string
Could not determine dtype for column 24, falling back to string
Could not determine dtype for column 34, falling back to string


In [611]:
adresy = (
    pl.scan_parquet("data/knihovny/adresy/*.parquet")
    .select(pl.col("I - NÁZEV KNIHOVNY", "search_query", "lon", "lat"))
    .collect()
)

do_grafu = (
    df.join(adresy, how="left", on="search_query")
    .filter(~pl.col("lon").is_null())
    .drop("I - NÁZEV KNIHOVNY_right")
    .with_row_index()
    .with_columns(
        pl.when(pl.col("index") == 4868)
        .then(pl.lit(True))
        .otherwise(pl.lit(False))
        .alias("Velenice")
    )
)

len(do_grafu)

5960

In [612]:
lon_min = do_grafu.select(pl.col("lon").min()).item()
lon_max = do_grafu.select(pl.col("lon").max()).item()
lat_min = do_grafu.select(pl.col("lat").min()).item()
lat_max = do_grafu.select(pl.col("lat").max()).item()

In [613]:
do_grafu.select(pl.col("J - druh knihovny").unique())

J - druh knihovny
str
"""základní"""
"""specializovaná"""
"""krajská knihovna"""
"""základní se specializovaným kn…"


In [614]:
do_grafu

index,I - NÁZEV KNIHOVNY,J - druh knihovny,ulice,psc,mesto,kraj,search_query,lon,lat,Velenice
u32,str,str,str,str,str,str,str,f64,f64,bool
0,"""Základní knihovna v Kralicích …","""základní""","""Masarykovo náměstí 41""","""798 12""","""Kralice na Hané""","""Olomoucký""","""Masarykovo náměstí 41, Kralice…",17.179025,49.463874,false
1,"""Základní knihovna v Lukavici""","""základní""","""Lukavice 1""","""538 21""","""Slatiňany""","""Pardubický""","""Lukavice 1, Slatiňany, 538 21,…",15.813254,49.921245,false
2,"""Studijní knihovna Muzea Beskyd…","""základní se specializovaným kn…","""Hluboká 66""","""738 01""","""Frýdek-Místek""","""Moravskoslezský""","""Hluboká 66, Frýdek-Místek, 738…",18.346858,49.687009,false
3,"""Městská knihovna Vladimíra Hol…","""základní""","""Masarykovo náměstí 140""","""294 21""","""Bělá pod Bezdězem""","""Středočeský""","""Masarykovo náměstí 140, Bělá p…",14.802146,50.501039,false
4,"""Městská knihovna Loket""","""základní""","""T. G. Masaryka 96""","""357 33""","""Loket""","""Karlovarský""","""T. G. Masaryka 96, Loket, 357 …",12.7555,50.186018,false
…,…,…,…,…,…,…,…,…,…,…
5955,"""Obecní knihovna Kynice""","""základní""","""Kynice 38""","""584 01""","""Kynice""","""Vysočina""","""Kynice 38, Kynice, 584 01, Vys…",15.360539,49.738956,false
5956,"""Obecní knihovna Radovesice""","""základní""","""Radovesice 43""","""410 02""","""Radovesice""","""Ústecký""","""Radovesice 43, Radovesice, 410…",14.074625,50.412784,false
5957,"""Obecní knihovna Poříčany""","""základní""","""Nádražní 90""","""289 14""","""Poříčany""","""Středočeský""","""Nádražní 90, Poříčany, 289 14,…",14.922498,50.110887,false


In [615]:
from math import cos, radians

In [616]:
def calculate_realistic_dimensions(lon_min, lon_max, lat_min, lat_max, width=600):
    """
    Calculate realistic chart height based on latitude to account for Earth's curvature.

    At different latitudes, 1 degree of longitude represents different distances:
    - At equator: ~111 km per degree
    - At 50°N: ~71 km per degree (111 * cos(50°))

    Args:
        lon_min, lon_max: Longitude range
        lat_min, lat_max: Latitude range
        width: Desired chart width in pixels

    Returns:
        height: Calculated height in pixels for realistic aspect ratio
    """
    # Use center latitude for calculation
    center_lat = (lat_min + lat_max) / 2

    # Calculate the actual distance ratio
    # 1 degree longitude at this latitude vs 1 degree latitude
    lon_scale_factor = cos(radians(center_lat))

    # Calculate ranges
    lon_range = lon_max - lon_min
    lat_range = lat_max - lat_min

    # Calculate height to maintain realistic proportions
    # height/width should equal (lat_range)/(lon_range * cos(lat))
    height = width * (lat_range / (lon_range * lon_scale_factor))

    return int(height)

In [617]:
chart_width = 600
chart_height = calculate_realistic_dimensions(
    lon_min, lon_max, lat_min, lat_max, chart_width
)

In [618]:
alt.Chart(do_grafu, width=chart_width, height=chart_height).mark_point(
    size=8, filled=True
).encode(
    alt.X("lon", scale=alt.Scale(domain=[lon_min, lon_max]), axis=None),
    alt.Y("lat", scale=alt.Scale(domain=[lat_min, lat_max]), axis=None),
    alt.Color("J - druh knihovny:N"),
)

alt.Chart(...)

In [619]:
from math import asin, sin, sqrt

import numpy as np

In [708]:
def haversine_distance(lon1, lat1, lon2, lat2):
    """
    Calculate the great circle distance in kilometers between two points
    on the earth (specified in decimal degrees)
    """
    # Convert decimal degrees to radians
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])

    # Haversine formula
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    c = 2 * asin(sqrt(a))
    r = 6371  # Radius of earth in kilometers
    return c * r


def find_furthest_point(df):
    """
    Find the library location that is furthest from all other locations.
    Returns the row index and minimum distance to nearest neighbor.
    """
    coords = df.select(["lon", "lat"]).to_numpy()
    n = len(coords)

    # For each point, find the minimum distance to any other point
    max_min_dist = -1
    furthest_idx = -1

    for i in range(n):
        min_dist = float("inf")
        for j in range(n):
            if i != j:
                dist = haversine_distance(
                    coords[i, 0], coords[i, 1], coords[j, 0], coords[j, 1]
                )
                min_dist = min(min_dist, dist)

        if min_dist > max_min_dist:
            max_min_dist = min_dist
            furthest_idx = i

    return furthest_idx, max_min_dist


def lat_lon_to_km_offset(lat_deg):
    """
    Calculate approximate km per degree at given latitude.
    Returns (km_per_degree_lon, km_per_degree_lat)
    """
    km_per_deg_lat = 111.32  # roughly constant
    km_per_deg_lon = 111.32 * cos(radians(lat_deg))
    return km_per_deg_lon, km_per_deg_lat


def find_densest_grid_point(df, grid_size_km=1.0, search_radius_km=2.0):
    """
    Create a grid with specified spacing and find the grid point with
    the most library locations within the search radius.

    Args:
        df: Polars dataframe with 'lon' and 'lat' columns
        grid_size_km: Grid cell size in kilometers (default 1.0)
        search_radius_km: Search radius in kilometers (default 2.0)

    Returns:
        Dictionary with grid point coordinates and count
    """
    coords = df.select(["lon", "lat"]).to_numpy()

    # Get bounds
    min_lon, max_lon = coords[:, 0].min(), coords[:, 0].max()
    min_lat, max_lat = coords[:, 1].min(), coords[:, 1].max()

    # Use center latitude for conversion
    center_lat = (min_lat + max_lat) / 2
    km_per_deg_lon, km_per_deg_lat = lat_lon_to_km_offset(center_lat)

    # Calculate grid spacing in degrees
    grid_spacing_lon = grid_size_km / km_per_deg_lon
    grid_spacing_lat = grid_size_km / km_per_deg_lat

    # Create grid
    lon_range = np.arange(min_lon, max_lon + grid_spacing_lon, grid_spacing_lon)
    lat_range = np.arange(min_lat, max_lat + grid_spacing_lat, grid_spacing_lat)

    max_count = 0
    best_grid_point = None

    # For each grid point, count libraries within search radius
    for grid_lon in lon_range:
        for grid_lat in lat_range:
            count = 0
            for lon, lat in coords:
                dist = haversine_distance(grid_lon, grid_lat, lon, lat)
                if dist <= search_radius_km:
                    count += 1

            if count > max_count:
                max_count = count
                best_grid_point = (grid_lon, grid_lat)

    return {
        "lon": best_grid_point[0],
        "lat": best_grid_point[1],
        "count": max_count,
        "grid_size_km": grid_size_km,
        "search_radius_km": search_radius_km,
    }


def find_furthest_point_stats(df):
    """
    Finds the library location furthest from its nearest neighbor,
    and calculates average/median distance of all nearest neighbors.

    Returns:
        tuple: (furthest_idx, max_min_dist, avg_dist, median_dist)
    """
    coords = df.select(["lon", "lat"]).to_numpy()
    n = len(coords)

    # List to store the distance to the nearest neighbor for every point
    nearest_neighbor_dists = []

    max_min_dist = -1
    furthest_idx = -1

    for i in range(n):
        min_dist = float("inf")

        # Find the nearest neighbor for point i
        for j in range(n):
            if i != j:
                dist = haversine_distance(
                    coords[i, 0], coords[i, 1], coords[j, 0], coords[j, 1]
                )
                min_dist = min(min_dist, dist)

        # Store this point's nearest neighbor distance
        nearest_neighbor_dists.append(min_dist)

        # Check if this is the new maximum (furthest point)
        if min_dist > max_min_dist:
            max_min_dist = min_dist
            furthest_idx = i

    # Calculate Statistics
    avg_dist = np.mean(nearest_neighbor_dists)
    median_dist = np.median(nearest_neighbor_dists)

    return furthest_idx, max_min_dist, avg_dist, median_dist

In [710]:
find_furthest_point_stats(do_grafu)

(4868, 10.121028843853589, 1.4272199165012487, 1.0699290337142089)

In [621]:
nejvzdalenejsi = find_furthest_point(do_grafu)
print(nejvzdalenejsi)

do_grafu.filter(pl.col("index") == nejvzdalenejsi[0])

(4868, 10.121028843853589)


index,I - NÁZEV KNIHOVNY,J - druh knihovny,ulice,psc,mesto,kraj,search_query,lon,lat,Velenice
u32,str,str,str,str,str,str,str,f64,f64,bool
4868,"""Městská knihovna České Velenic…","""základní""","""Na Sadech 166""","""378 10""","""České Velenice""","""Jihočeský""","""Na Sadech 166, České Velenice,…",14.963881,48.769143,true


Nejblíž: by oko Nové Hrady.

In [622]:
def find_densest_grid_point(df, grid_size_km=1.0, search_radius_km=2.0, verbose=True):
    """
    Create a grid with specified spacing and find the grid point with
    the most library locations within the search radius.

    Args:
        df: Polars dataframe with 'lon' and 'lat' columns
        grid_size_km: Grid cell size in kilometers (default 1.0)
        search_radius_km: Search radius in kilometers (default 2.0)

    Returns:
        Dictionary with grid point coordinates and count
    """
    coords = df.select(["lon", "lat"]).to_numpy()

    # Get bounds
    min_lon, max_lon = coords[:, 0].min(), coords[:, 0].max()
    min_lat, max_lat = coords[:, 1].min(), coords[:, 1].max()

    # Use center latitude for conversion
    center_lat = (min_lat + max_lat) / 2
    km_per_deg_lon, km_per_deg_lat = lat_lon_to_km_offset(center_lat)

    # Calculate grid spacing in degrees
    grid_spacing_lon = grid_size_km / km_per_deg_lon
    grid_spacing_lat = grid_size_km / km_per_deg_lat

    # Create grid
    lon_range = np.arange(min_lon, max_lon + grid_spacing_lon, grid_spacing_lon)
    lat_range = np.arange(min_lat, max_lat + grid_spacing_lat, grid_spacing_lat)

    max_count = 0
    best_grid_point = None

    # For each grid point, count libraries within search radius

    print(f"no of lons: {len(lon_range)}")
    pocitadlo = 0

    for grid_lon in lon_range:
        pocitadlo += 1
        if pocitadlo % 10 == 0:
            if verbose == True:
                print(f"{pocitadlo}/{len(lon_range)}")
        for grid_lat in lat_range:
            count = 0
            for lon, lat in coords:
                dist = haversine_distance(grid_lon, grid_lat, lon, lat)
                if dist <= search_radius_km:
                    count += 1

            if count > max_count:
                max_count = count
                best_grid_point = (grid_lon, grid_lat)

    return {
        "lon": best_grid_point[0],
        "lat": best_grid_point[1],
        "count": max_count,
        "grid_size_km": grid_size_km,
        "search_radius_km": search_radius_km,
    }

In [623]:
import polars as pl


def haversine_vectorized(lat1, lon1, lat2_array, lon2_array):
    """
    Calculate haversine distance from a single point (lat1, lon1)
    to an array of points (lat2_array, lon2_array).
    """
    R = 6371.0  # Earth radius in kilometers

    dlat = np.radians(lat2_array - lat1)
    dlon = np.radians(lon2_array - lon1)

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(np.radians(lat1))
        * np.cos(np.radians(lat2_array))
        * np.sin(dlon / 2) ** 2
    )

    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c


def analyze_grid_density(
    df, grid_size_km=1.0, search_radii_km=[1.0, 2.0], verbose=True
):
    """
    Create a grid and analyze point density and proximity for every grid node.

    Args:
        df: Polars dataframe with 'lon' and 'lat' columns.
        grid_size_km: Grid cell size in kilometers.
        search_radii_km: List of radii (floats) to count points within.
        verbose: Print progress.

    Returns:
        List of dictionaries containing coordinates, counts for each radius,
        and distance to the nearest point.
    """
    # Convert dataframe columns to numpy arrays for vectorization
    points_lon = df["lon"].to_numpy()
    points_lat = df["lat"].to_numpy()

    # --- Grid Generation (Logic preserved from original) ---
    min_lon, max_lon = points_lon.min(), points_lon.max()
    min_lat, max_lat = points_lat.min(), points_lat.max()

    # Estimate degrees per km based on center latitude
    center_lat = (min_lat + max_lat) / 2

    # Approximation: 1 deg lat ~= 111km; 1 deg lon ~= 111km * cos(lat)
    km_per_deg_lat = 111.32
    km_per_deg_lon = 111.32 * np.cos(np.radians(center_lat))

    grid_spacing_lon = grid_size_km / km_per_deg_lon
    grid_spacing_lat = grid_size_km / km_per_deg_lat

    lon_range = np.arange(min_lon, max_lon + grid_spacing_lon, grid_spacing_lon)
    lat_range = np.arange(min_lat, max_lat + grid_spacing_lat, grid_spacing_lat)

    results = []
    total_grid_points = len(lon_range) * len(lat_range)

    if verbose:
        print(
            f"Grid dimensions: {len(lon_range)} x {len(lat_range)} ({total_grid_points} points)"
        )
        print(f"Calculating for radii: {search_radii_km}")

    processed_count = 0

    # --- Analysis Loop ---
    for grid_lon in lon_range:
        for grid_lat in lat_range:
            # 1. Calculate distances to ALL points at once (Vectorized)
            # This replaces the innermost loop over coords
            dists = haversine_vectorized(grid_lat, grid_lon, points_lat, points_lon)

            # 2. Find distance to nearest point
            # If dists is empty (no points in df), set to None or inf
            nearest_dist = dists.min() if len(dists) > 0 else float("inf")

            # 3. Create the result dictionary for this grid point
            row_data = {
                "grid_lat": grid_lat,
                "grid_lon": grid_lon,
                "nearest_point_dist_km": nearest_dist,
            }

            # 4. Calculate counts for each requested radius
            for r in search_radii_km:
                # Count how many distances are less than radius r
                count = np.sum(dists <= r)
                row_data[f"count_within_{r}km"] = count

            results.append(row_data)

            processed_count += 1

        # Verbose logging (per longitude column processed)
        if verbose and (processed_count % (len(lat_range) * 5) == 0):
            print(f"Processed: {processed_count}/{total_grid_points} grid points")

    return results

In [624]:
import geopandas as gpd
from shapely.geometry import Point
from shapely.prepared import prep

In [625]:
def haversine_vectorized(lat1, lon1, lat2_array, lon2_array):
    """
    Calculate haversine distance from a single point (lat1, lon1)
    to an array of points (lat2_array, lon2_array).
    """
    R = 6371.0  # Earth radius in kilometers

    dlat = np.radians(lat2_array - lat1)
    dlon = np.radians(lon2_array - lon1)

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(np.radians(lat1))
        * np.cos(np.radians(lat2_array))
        * np.sin(dlon / 2) ** 2
    )

    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c


def analyze_grid_density(
    df,
    boundary_geometry=None,
    grid_size_km=1.0,
    search_radii_km=[1.0, 2.0],
    verbose=True,
):
    """
    Create a grid and analyze point density, restricting grid nodes to a specific boundary.

    Args:
        df: Polars dataframe with 'lon' and 'lat' columns.
        boundary_geometry: A shapely Polygon or MultiPolygon defining the border (e.g., Czech Republic).
        grid_size_km: Grid cell size in kilometers.
        search_radii_km: List of radii (floats) to count points within.
        verbose: Print progress.
    """
    # 1. Prepare Boundary for Fast Searching
    # Using 'prep' speeds up the "contains" check significantly for complex borders
    prepared_boundary = None
    if boundary_geometry:
        prepared_boundary = prep(boundary_geometry)

    # Convert dataframe columns to numpy arrays for vectorization
    points_lon = df["lon"].to_numpy()
    points_lat = df["lat"].to_numpy()

    # --- Grid Generation ---
    min_lon, max_lon = points_lon.min(), points_lon.max()
    min_lat, max_lat = points_lat.min(), points_lat.max()

    center_lat = (min_lat + max_lat) / 2

    km_per_deg_lat = 111.32
    km_per_deg_lon = 111.32 * np.cos(np.radians(center_lat))

    grid_spacing_lon = grid_size_km / km_per_deg_lon
    grid_spacing_lat = grid_size_km / km_per_deg_lat

    lon_range = np.arange(min_lon, max_lon + grid_spacing_lon, grid_spacing_lon)
    lat_range = np.arange(min_lat, max_lat + grid_spacing_lat, grid_spacing_lat)

    results = []
    total_potential_points = len(lon_range) * len(lat_range)

    if verbose:
        print(
            f"Grid bounding box: {len(lon_range)} x {len(lat_range)} potential points"
        )
        print(f"Filtering by boundary: {'Yes' if boundary_geometry else 'No'}")

    processed_count = 0

    # --- Analysis Loop ---
    for grid_lon in lon_range:
        for grid_lat in lat_range:
            # --- NEW: Boundary Check ---
            # If a boundary is provided, skip points outside of it
            if prepared_boundary:
                # Create a point for the current grid node
                pt = Point(grid_lon, grid_lat)
                if not prepared_boundary.contains(pt):
                    continue

            # 1. Calculate distances (Vectorized)
            dists = haversine_vectorized(grid_lat, grid_lon, points_lat, points_lon)

            # 2. Find distance to nearest point
            nearest_dist = dists.min() if len(dists) > 0 else float("inf")

            # 3. Create result dict
            row_data = {
                "grid_lat": grid_lat,
                "grid_lon": grid_lon,
                "nearest_point_dist_km": nearest_dist,
            }

            # 4. Calculate counts
            for r in search_radii_km:
                count = np.sum(dists <= r)
                row_data[f"count_within_{r}km"] = count

            results.append(row_data)

            processed_count += 1

        # Verbose logging
        if verbose and (processed_count % 1000 == 0):
            print(f"Processed valid grid points: {processed_count}")

    return results

In [626]:
world = gpd.read_file(gpd.datasets.get_path("naturalearth_lowres"))
cz_boundary = world[world.name == "Czechia"].geometry.iloc[0]

C:\Users\micha\AppData\Local\Temp\ipykernel_33892\3933795955.py:1: FutureWarning: The geopandas.dataset module is deprecated and will be removed in GeoPandas 1.0. You can get the original 'naturalearth_lowres' data from https://www.naturalearthdata.com/downloads/110m-cultural-vectors/.
  world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))


In [639]:
republika = analyze_grid_density(
    do_grafu,
    grid_size_km=0.2,
    search_radii_km=[1, 2, 5, 10],
    verbose=True,
    boundary_geometry=cz_boundary,
)
republika = pl.DataFrame(republika).with_row_index()
republika.write_parquet("data/knihovny/mrizka_republika.parquet")

Grid bounding box: 2386 x 1343 potential points
Filtering by boundary: Yes
Processed valid grid points: 0
Processed valid grid points: 0
Processed valid grid points: 0
Processed valid grid points: 0
Processed valid grid points: 0
Processed valid grid points: 0
Processed valid grid points: 0
Processed valid grid points: 0
Processed valid grid points: 0
Processed valid grid points: 0
Processed valid grid points: 0
Processed valid grid points: 0
Processed valid grid points: 0
Processed valid grid points: 0
Processed valid grid points: 0
Processed valid grid points: 0
Processed valid grid points: 0
Processed valid grid points: 0
Processed valid grid points: 0
Processed valid grid points: 0
Processed valid grid points: 0
Processed valid grid points: 0
Processed valid grid points: 0
Processed valid grid points: 0
Processed valid grid points: 0
Processed valid grid points: 115000
Processed valid grid points: 141000


In [640]:
republika.select(pl.col("nearest_point_dist_km").median())

nearest_point_dist_km
f64
2.090581


In [641]:
republika.select(pl.col("count_within_5km").median())

count_within_5km
f64
5.0


In [642]:
republika.sort(by="nearest_point_dist_km").tail(10)

index,grid_lat,grid_lon,nearest_point_dist_km,count_within_1km,count_within_2km,count_within_5km,count_within_10km
u32,f64,f64,f64,i64,i64,i64,i64
1562447,50.289045,16.768953,19.191226,0,0,0,0
1562448,50.290842,16.768953,19.20463,0,0,0,0
1561547,50.294435,16.766169,19.23581,0,0,0,0
1559748,50.285452,16.7606,19.297674,0,0,0,0
1560644,50.287249,16.763384,19.308157,0,0,0,0
1561546,50.292639,16.766169,19.310457,0,0,0,0
1561544,50.289045,16.766169,19.322741,0,0,0,0
1561545,50.290842,16.766169,19.386875,0,0,0,0
1560645,50.289045,16.763384,19.455383,0,0,0,0


In [750]:
republika.filter(pl.col("count_within_5km") == 35).sample(40).sort(
    by="nearest_point_dist_km"
).tail(40).head(10)

index,grid_lat,grid_lon,nearest_point_dist_km,count_within_1km,count_within_2km,count_within_5km,count_within_10km
u32,f64,f64,f64,i64,i64,i64,i64
678116,50.145316,14.444051,0.152163,1,4,35,186
647306,50.138129,14.380012,0.219429,2,5,35,178
629810,50.071654,14.343816,0.262422,2,4,35,167
678115,50.143519,14.444051,0.299,1,4,35,186
704435,50.066264,14.499738,0.31713,2,3,35,179
625791,50.08962,14.335463,0.390954,2,4,35,172
704434,50.064468,14.499738,0.510695,2,3,35,179
647303,50.132739,14.380012,0.527944,5,5,35,179
697874,50.044705,14.485816,0.601117,3,5,35,171


In [752]:
republika.filter(pl.col("count_within_5km") == 40).sample(40).sort(
    by="nearest_point_dist_km", descending=True
).head(10)

index,grid_lat,grid_lon,nearest_point_dist_km,count_within_1km,count_within_2km,count_within_5km,count_within_10km
u32,f64,f64,f64,i64,i64,i64,i64
1528341,49.214665,16.660365,2.899701,0,0,40,81
1528340,49.212869,16.660365,2.744717,0,0,40,82
1516474,49.15897,16.621384,2.648998,0,0,40,91
1514783,49.157173,16.615816,2.581007,0,0,40,91
1519011,49.162563,16.629737,2.530933,0,0,40,90
1515629,49.15897,16.6186,2.524242,0,0,40,90
1491156,49.187716,16.537855,2.486369,0,0,40,80
1514829,49.239818,16.615816,2.431008,0,0,40,78
663364,50.035722,14.413424,2.26868,0,0,40,164


In [643]:
republika.sort(by="count_within_5km").tail(10)

index,grid_lat,grid_lon,nearest_point_dist_km,count_within_1km,count_within_2km,count_within_5km,count_within_10km
u32,f64,f64,f64,i64,i64,i64,i64
670091,50.093213,14.427345,0.134412,22,72,136,193
670092,50.09501,14.427345,0.313064,20,71,136,193
671428,50.09501,14.43013,0.319166,14,67,136,192
680744,50.093213,14.44962,0.730878,2,25,136,193
664738,50.09501,14.416208,0.501248,22,68,137,192
664739,50.096807,14.416208,0.618861,14,67,137,191
664740,50.098603,14.416208,0.622311,9,68,137,190
666079,50.096807,14.418992,0.423294,15,66,137,192
666080,50.098603,14.418992,0.428332,9,64,137,193


In [644]:
len(republika)

2017654

In [671]:
rozhodna_vzdalenost = 7
zoom = 1.3

alt.Chart(
    republika.filter(pl.col("nearest_point_dist_km") > rozhodna_vzdalenost).sample(
        5000
    ),
    width=chart_width * zoom,
    height=chart_height * zoom,
).mark_point(size=10, filled=True).encode(
    alt.X("grid_lon", scale=alt.Scale(domain=[lon_min, lon_max]), axis=None),
    alt.Y("grid_lat", scale=alt.Scale(domain=[lat_min, lat_max]), axis=None),
)

alt.Chart(...)

In [673]:
rozhodna_vzdalenost = 8
zoom = 1.3

alt.Chart(
    republika.filter(pl.col("nearest_point_dist_km") > rozhodna_vzdalenost).sample(
        5000
    ),
    width=chart_width * zoom,
    height=chart_height * zoom,
).mark_point(size=10, filled=True).encode(
    alt.X("grid_lon", scale=alt.Scale(domain=[lon_min, lon_max]), axis=None),
    alt.Y("grid_lat", scale=alt.Scale(domain=[lat_min, lat_max]), axis=None),
)

alt.Chart(...)

In [675]:
rozhodna_vzdalenost = 10
zoom = 1.3

alt.Chart(
    republika.filter(pl.col("nearest_point_dist_km") > rozhodna_vzdalenost).sample(
        5000
    ),
    width=chart_width * zoom,
    height=chart_height * zoom,
).mark_point(size=10, filled=True).encode(
    alt.X("grid_lon", scale=alt.Scale(domain=[lon_min, lon_max]), axis=None),
    alt.Y("grid_lat", scale=alt.Scale(domain=[lat_min, lat_max]), axis=None),
)

alt.Chart(...)

In [677]:
rozhodna_vzdalenost = 10
zoom = 15

alt.Chart(
    republika.filter(pl.col("nearest_point_dist_km") > rozhodna_vzdalenost).sample(
        5000
    ),
    width=chart_width * zoom,
    height=chart_height * zoom,
).mark_point(size=10, filled=True).encode(
    alt.X("grid_lon", scale=alt.Scale(domain=[lon_min, lon_max]), axis=None),
    alt.Y("grid_lat", scale=alt.Scale(domain=[lat_min, lat_max]), axis=None),
) + alt.Chart(
    republika.filter(pl.col("nearest_point_dist_km") > rozhodna_vzdalenost),
    width=chart_width * zoom,
    height=chart_height * zoom,
).mark_text(size=5).encode(
    alt.X("grid_lon", scale=alt.Scale(domain=[lon_min, lon_max]), axis=None),
    alt.Y("grid_lat", scale=alt.Scale(domain=[lat_min, lat_max]), axis=None),
    alt.Text("index:N"),
)

alt.LayerChart(...)

In [767]:
# Create the base chart with a standard viewing size (e.g., 800x600)
base = (
    alt.Chart(
        republika.filter(pl.col("count_within_5km").is_between(21, 25)).sample(100)
    )
    .encode(
        alt.X("grid_lon", scale=alt.Scale(domain=[lon_min, lon_max]), axis=None),
        alt.Y("grid_lat", scale=alt.Scale(domain=[lat_min, lat_max]), axis=None),
    )
    .properties(
        width=800,  # The viewing window size
        height=600,
    )
)

# Layer 1: The points
points = base.mark_point(size=10, filled=True)

# Layer 2: The text labels
text = base.mark_text(size=5).encode(alt.Text("index:N"))

# Combine and enable ZOOM/PAN
zoomable_chart = (points + text).interactive()

zoomable_chart

alt.LayerChart(...)

In [779]:
republika.filter(pl.col("index") == 1584479)

index,grid_lat,grid_lon,nearest_point_dist_km,count_within_1km,count_within_2km,count_within_5km,count_within_10km
u32,f64,f64,f64,i64,i64,i64,i64
1584479,49.178733,16.835776,1.455402,0,1,22,39


In [777]:
republika.filter(pl.col("index") == 234432)

index,grid_lat,grid_lon,nearest_point_dist_km,count_within_1km,count_within_2km,count_within_5km,count_within_10km
u32,f64,f64,f64,i64,i64,i64,i64
234432,49.777008,13.413855,2.841286,0,0,22,36


In [773]:
republika.filter(pl.col("index") == 702501)

index,grid_lat,grid_lon,nearest_point_dist_km,count_within_1km,count_within_2km,count_within_5km,count_within_10km
u32,f64,f64,f64,i64,i64,i64,i64
702501,48.939782,14.496953,1.405075,0,4,21,41


In [771]:
republika.filter(pl.col("index") == 1792855)

index,grid_lat,grid_lon,nearest_point_dist_km,count_within_1km,count_within_2km,count_within_5km,count_within_10km
u32,f64,f64,f64,i64,i64,i64,i64
1792855,49.473379,17.456678,1.582476,0,4,21,50


In [769]:
republika.filter(pl.col("index") == 1077161)

index,grid_lat,grid_lon,nearest_point_dist_km,count_within_1km,count_within_2km,count_within_5km,count_within_10km
u32,f64,f64,f64,i64,i64,i64,i64
1077161,49.920738,15.399071,0.68417,2,17,21,37


In [679]:
# Create the base chart with a standard viewing size (e.g., 800x600)
base = (
    alt.Chart(republika.filter(pl.col("nearest_point_dist_km") > 7).sample(5000))
    .encode(
        alt.X("grid_lon", scale=alt.Scale(domain=[lon_min, lon_max]), axis=None),
        alt.Y("grid_lat", scale=alt.Scale(domain=[lat_min, lat_max]), axis=None),
    )
    .properties(
        width=800,  # The viewing window size
        height=600,
    )
)

# Layer 1: The points
points = base.mark_point(size=10, filled=True)

# Layer 2: The text labels
text = base.mark_text(size=5).encode(alt.Text("index:N"))

# Combine and enable ZOOM/PAN
zoomable_chart = (points + text).interactive()

zoomable_chart

alt.LayerChart(...)

(Hradisko, Ralsko, Libavá)

In [718]:
republika.filter(pl.col("index") == 843792)

index,grid_lat,grid_lon,nearest_point_dist_km,count_within_1km,count_within_2km,count_within_5km,count_within_10km
u32,f64,f64,f64,i64,i64,i64,i64
843792,50.590878,14.811581,8.79198,0,0,0,5


In [687]:
republika.sort(by="count_within_5km")

index,grid_lat,grid_lon,nearest_point_dist_km,count_within_1km,count_within_2km,count_within_5km,count_within_10km
u32,f64,f64,f64,i64,i64,i64,i64
0,50.263893,12.24166,5.24126,0,0,0,4
1,50.265689,12.24166,5.310342,0,0,0,4
2,50.260299,12.244444,5.315157,0,0,0,4
3,50.262096,12.244444,5.368435,0,0,0,4
4,50.263893,12.244444,5.428547,0,0,0,4
…,…,…,…,…,…,…,…
664739,50.096807,14.416208,0.618861,14,67,137,191
664740,50.098603,14.416208,0.622311,9,68,137,190
666079,50.096807,14.418992,0.423294,15,66,137,192


In [714]:
republika.sort(by="count_within_10km")

index,grid_lat,grid_lon,nearest_point_dist_km,count_within_1km,count_within_2km,count_within_5km,count_within_10km
u32,f64,f64,f64,i64,i64,i64,i64
6771,49.929721,12.425424,10.706002,0,0,0,0
6772,49.931517,12.425424,10.511268,0,0,0,0
6773,49.933314,12.425424,10.316727,0,0,0,0
6774,49.935111,12.425424,10.122389,0,0,0,0
6990,49.918941,12.428209,11.919263,0,0,0,0
…,…,…,…,…,…,…,…
683416,50.129146,14.455189,0.233844,7,16,45,194
684735,50.120163,14.457973,0.426045,12,16,82,194
686057,50.118366,14.460757,0.175555,11,16,85,194


In [689]:
do_grafu.columns

['index',
 'I - NÁZEV KNIHOVNY',
 'J - druh knihovny',
 'ulice',
 'psc',
 'mesto',
 'kraj',
 'search_query',
 'lon',
 'lat',
 'Velenice']

In [691]:
do_grafu.select(pl.col("kraj").unique()).to_series().to_list()

['jihočeský',
 'Pardubický',
 'Praha',
 'Olomoucký',
 'Ústecký',
 'Královéhradecký',
 'Zlínský',
 'JIhomoravský',
 'Plzeňský',
 'Liberecký',
 'Hlavní město Praha',
 'Moravskoslezský',
 'královéhradecký',
 'Středočeský',
 'Hradec Králové',
 'Plzeńský',
 'Karlovarský',
 'Vysočina',
 'Jihočeský',
 'Jihomoravský']

In [693]:
praha_stredocesky = analyze_grid_density(
    do_grafu.filter(
        (pl.col("mesto").str.contains("Praha"))
        | (pl.col("kraj").str.contains("ředočes"))
    ),
    grid_size_km=0.2,
    search_radii_km=[7, 10],
    verbose=False,
    boundary_geometry=cz_boundary,
)

In [694]:
pl.DataFrame(praha_stredocesky).sort(by="count_within_10km")

grid_lat,grid_lon,nearest_point_dist_km,count_within_7km,count_within_10km
f64,f64,f64,i64,i64
49.521664,13.470799,23.577627,0,0
49.523461,13.470799,23.570702,0,0
49.525258,13.470799,23.565469,0,0
49.527054,13.470799,23.561928,0,0
49.528851,13.470799,23.560081,0,0
…,…,…,…,…
50.11994,14.458365,0.389888,149,194
50.118143,14.461163,0.142511,150,194
50.091194,14.419198,0.07272,159,195


In [695]:
alt.Chart(
    do_grafu.filter(
        (pl.col("mesto").str.contains("Praha"))
        | (pl.col("kraj").str.contains("ředočes"))
    )
).mark_point().encode(
    alt.X("lon", scale=alt.Scale(domain=[lon_min, lon_max]), axis=None),
    alt.Y("lat", scale=alt.Scale(domain=[lat_min, lat_max]), axis=None),
    alt.Color("kraj:N"),
)

alt.Chart(...)

In [696]:
v_praze = analyze_grid_density(
    do_grafu.filter(pl.col("mesto").str.contains("Praha")),
    grid_size_km=0.025,
    search_radii_km=[0.2, 0.5, 1, 2, 3],
    verbose=True,
    boundary_geometry=cz_boundary,
)

Grid bounding box: 1264 x 1252 potential points
Filtering by boundary: Yes
Processed valid grid points: 313000
Processed valid grid points: 626000
Processed valid grid points: 939000
Processed valid grid points: 1252000
Processed valid grid points: 1565000


In [697]:
pl.DataFrame(v_praze).sort(by="count_within_0.2km")

grid_lat,grid_lon,nearest_point_dist_km,count_within_0.2km,count_within_0.5km,count_within_1km,count_within_2km,count_within_3km
f64,f64,f64,i64,i64,i64,i64,i64
49.959749,14.261764,6.620765,0,0,0,0,0
49.959973,14.261764,6.620797,0,0,0,0,0
49.960198,14.261764,6.620922,0,0,0,0,0
49.960422,14.261764,6.621142,0,0,0,0,0
49.960647,14.261764,6.621457,0,0,0,0,0
…,…,…,…,…,…,…,…
50.088656,14.417564,0.111587,8,18,48,82,100
50.085961,14.417214,0.11618,9,25,47,84,101
50.085737,14.416863,0.126923,10,25,49,84,101


In [698]:
pl.DataFrame(v_praze).sort(by="count_within_1km")

grid_lat,grid_lon,nearest_point_dist_km,count_within_0.2km,count_within_0.5km,count_within_1km,count_within_2km,count_within_3km
f64,f64,f64,i64,i64,i64,i64,i64
49.959749,14.261764,6.620765,0,0,0,0,0
49.959973,14.261764,6.620797,0,0,0,0,0
49.960198,14.261764,6.620922,0,0,0,0,0
49.960422,14.261764,6.621142,0,0,0,0,0
49.960647,14.261764,6.621457,0,0,0,0,0
…,…,…,…,…,…,…,…
50.083042,14.423516,0.318958,0,12,54,84,101
50.082593,14.423866,0.285193,0,12,54,84,101
50.082817,14.423866,0.288994,0,12,54,84,101


In [699]:
pl.DataFrame(v_praze).write_parquet("data/knihovny/mrizka_praha.parquet")